In [33]:
import pandas as pd

product_features = pd.read_csv(
    "../data/processed/product_clusters_v2.csv",
    index_col=0
)

product_features.index.name = "StockCode"
product_features = product_features.reset_index()
product_features.head()

,StockCode,total_quantity,avg_quantity_per_order,related_product_count,cluster,cluster_name
0,71,860,12.112676,2067,2,High-Volume Products
1,22,303,13.772727,908,0,Active/Regular Products
2,29,192,6.620690,1001,0,Active/Regular Products
3,3,5,1.666667,85,1,Slow-Moving Products
4,5,16,3.200000,196,1,Slow-Moving Products


In [34]:
zone_mapping = {
    "High-Volume Products": "Zone A",
    "Active/Regular Products": "Zone B",
    "Low-Movement Products": "Zone C",
    "Slow-Moving Products": "Zone D"
}

In [35]:
product_features["recommended_zone"] = (
    product_features["cluster_name"].map(zone_mapping)
)

In [36]:
product_features[
    ["StockCode", "cluster_name", "recommended_zone"]
].head(20)

,StockCode,cluster_name,recommended_zone
0,71,High-Volume Products,Zone A
1,22,Active/Regular Products,Zone B
2,29,Active/Regular Products,Zone B
3,3,Slow-Moving Products,Zone D
4,5,Slow-Moving Products,Zone D
5,4,Slow-Moving Products,Zone D
6,91,High-Volume Products,Zone A
7,196,High-Volume Products,Zone A
8,175,High-Volume Products,Zone A
9,113,High-Volume Products,Zone A


In [37]:
zone_distance = {
    "Zone A": 10,
    "Zone B": 25,
    "Zone C": 40,
    "Zone D": 60
}

In [38]:
product_features["distance_from_packing"] = (
    product_features["recommended_zone"].map(zone_distance)
)

In [39]:
product_features[
    [
        "StockCode",
        "cluster_name",
        "recommended_zone",
        "distance_from_packing"
    ]
].head(20)

,StockCode,cluster_name,recommended_zone,distance_from_packing
0,71,High-Volume Products,Zone A,10
1,22,Active/Regular Products,Zone B,25
2,29,Active/Regular Products,Zone B,25
3,3,Slow-Moving Products,Zone D,60
4,5,Slow-Moving Products,Zone D,60
5,4,Slow-Moving Products,Zone D,60
6,91,High-Volume Products,Zone A,10
7,196,High-Volume Products,Zone A,10
8,175,High-Volume Products,Zone A,10
9,113,High-Volume Products,Zone A,10


In [56]:
df = pd.read_csv(
    "../data/processed/cleaned_online_retail.csv"
)
df["StockCode"] = df["StockCode"].astype(str)
product_features["StockCode"] = product_features["StockCode"].astype(str)

C:\Users\HP\AppData\Local\Temp\ipykernel_40140\1697342282.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [57]:
order_frequency = (
    df.groupby("StockCode")["InvoiceNo"]
      .nunique()
      .rename("order_frequency")
)

In [58]:
order_frequency.head()

StockCode
10002     71
10080     22
10120     29
10123C     3
10124A     5
Name: order_frequency, dtype: int64

In [59]:
product_features = product_features.merge(
    order_frequency,
    on="StockCode",
    how="left"
)

In [60]:
print(product_features.columns.tolist())

['StockCode', 'total_quantity', 'avg_quantity_per_order', 'related_product_count', 'cluster', 'cluster_name', 'recommended_zone', 'distance_from_packing', 'order_frequency']


In [61]:
product_features["order_frequency"].isna().sum()

np.int64(3922)

In [62]:
product_features["order_frequency"] = (
    product_features["order_frequency"].fillna(0)
)

In [63]:
product_features["picking_priority"] = (
    product_features["order_frequency"] *
    product_features["total_quantity"]
)

In [64]:
top_products = product_features.sort_values(
    "picking_priority",
    ascending=False
)

top_products[
    [
        "StockCode",
        "order_frequency",
        "total_quantity",
        "cluster_name",
        "recommended_zone",
        "picking_priority"
    ]
].head(10)

,StockCode,order_frequency,total_quantity,cluster_name,recommended_zone,picking_priority
0,71,0.0,860,High-Volume Products,Zone A,0.0
1,22,0.0,303,Active/Regular Products,Zone B,0.0
2,29,0.0,192,Active/Regular Products,Zone B,0.0
3,3,0.0,5,Slow-Moving Products,Zone D,0.0
4,5,0.0,16,Slow-Moving Products,Zone D,0.0
5,4,0.0,17,Slow-Moving Products,Zone D,0.0
6,91,0.0,1295,High-Volume Products,Zone A,0.0
7,196,0.0,2856,High-Volume Products,Zone A,0.0
8,175,0.0,2229,High-Volume Products,Zone A,0.0
9,113,0.0,1615,High-Volume Products,Zone A,0.0


In [65]:
product_features["current_distance"] = 60

In [66]:
product_features["distance_saved"] = (
    product_features["current_distance"]
    - product_features["distance_from_packing"]
)

In [67]:
product_features[
    [
        "StockCode",
        "current_distance",
        "distance_from_packing",
        "distance_saved"
    ]
].head(20)

,StockCode,current_distance,distance_from_packing,distance_saved
0,71,60,10,50
1,22,60,25,35
2,29,60,25,35
3,3,60,60,0
4,5,60,60,0
5,4,60,60,0
6,91,60,10,50
7,196,60,10,50
8,175,60,10,50
9,113,60,10,50


In [68]:
total_current_distance = (
    product_features["current_distance"]
    * product_features["order_frequency"]
).sum()

In [69]:
total_recommended_distance = (
    product_features["distance_from_packing"]
    * product_features["order_frequency"]
).sum()

In [70]:
travel_reduction = (
    (total_current_distance - total_recommended_distance)
    / total_current_distance
) * 100

C:\Users\HP\AppData\Local\Temp\ipykernel_40140\928008467.py:2: RuntimeWarning: invalid value encountered in scalar divide
  (total_current_distance - total_recommended_distance)


In [71]:
print(
    f"Estimated travel reduction: {travel_reduction:.2f}%"
)

Estimated travel reduction: nan%


In [72]:
product_features[
    [
        "order_frequency",
        "current_distance",
        "distance_from_packing"
    ]
].isna().sum()

order_frequency          0
current_distance         0
distance_from_packing    0
dtype: int64

In [73]:
print("Current distance:", total_current_distance)
print("Recommended distance:", total_recommended_distance)
print("Travel reduction:", travel_reduction)

Current distance: 0.0
Recommended distance: 0.0
Travel reduction: nan


In [74]:
print(
    product_features[
        [
            "order_frequency",
            "current_distance",
            "distance_from_packing"
        ]
    ].dtypes
)

order_frequency          float64
current_distance           int64
distance_from_packing      int64
dtype: object


In [75]:
product_features["order_frequency"] = pd.to_numeric(
    product_features["order_frequency"],
    errors="coerce"
)

product_features["current_distance"] = pd.to_numeric(
    product_features["current_distance"],
    errors="coerce"
)

product_features["distance_from_packing"] = pd.to_numeric(
    product_features["distance_from_packing"],
    errors="coerce"
)

In [76]:
print(
    product_features[
        [
            "order_frequency",
            "current_distance",
            "distance_from_packing"
        ]
    ].isna().sum()
)

order_frequency          0
current_distance         0
distance_from_packing    0
dtype: int64


In [77]:
total_current_distance = (
    product_features["current_distance"]
    * product_features["order_frequency"]
).sum()

total_recommended_distance = (
    product_features["distance_from_packing"]
    * product_features["order_frequency"]
).sum()

print("Current:", total_current_distance)
print("Recommended:", total_recommended_distance)

Current: 0.0
Recommended: 0.0


In [78]:
travel_reduction = (
    (total_current_distance - total_recommended_distance)
    / total_current_distance
) * 100

print(f"Estimated travel reduction: {travel_reduction:.2f}%")

Estimated travel reduction: nan%


C:\Users\HP\AppData\Local\Temp\ipykernel_40140\1085259812.py:2: RuntimeWarning: invalid value encountered in scalar divide
  (total_current_distance - total_recommended_distance)


In [79]:
print("Current:", total_current_distance)
print("Recommended:", total_recommended_distance)
print(product_features[
    ["order_frequency", "current_distance", "distance_from_packing"]
].dtypes)

Current: 0.0
Recommended: 0.0
order_frequency          float64
current_distance           int64
distance_from_packing      int64
dtype: object


In [80]:
df["StockCode"] = (
    df["StockCode"]
    .astype(str)
    .str.strip()
)

product_features["StockCode"] = (
    product_features["StockCode"]
    .astype(str)
    .str.strip()
)

In [81]:
order_frequency = (
    df.groupby("StockCode")["InvoiceNo"]
      .nunique()
      .reset_index(name="order_frequency")
)

In [82]:
print(order_frequency.head())
print(order_frequency["order_frequency"].describe())

  StockCode  order_frequency
0     10002               71
1     10080               22
2     10120               29
3    10123C                3
4    10124A                5
count    3922.000000
mean      132.484192
std       193.890513
min         1.000000
25%        16.000000
50%        65.000000
75%       165.000000
max      2198.000000
Name: order_frequency, dtype: float64


In [83]:
if "order_frequency" in product_features.columns:
    product_features = product_features.drop(
        columns=["order_frequency"]
    )

In [84]:
product_features = product_features.merge(
    order_frequency,
    on="StockCode",
    how="left"
)

In [85]:
print(product_features["order_frequency"].describe())

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: order_frequency, dtype: float64


In [86]:
print(
    product_features[
        ["StockCode", "order_frequency"]
    ].head(20)
)

   StockCode  order_frequency
0         71              NaN
1         22              NaN
2         29              NaN
3          3              NaN
4          5              NaN
5          4              NaN
6         91              NaN
7        196              NaN
8        175              NaN
9        113              NaN
10        12              NaN
11       138              NaN
12       518              NaN
13       145              NaN
14       102              NaN
15        60              NaN
16        88              NaN
17        84              NaN
18       321              NaN
19       456              NaN


In [87]:
print(product_features.head())

  StockCode  total_quantity  avg_quantity_per_order  related_product_count  \
0        71             860               12.112676                   2067   
1        22             303               13.772727                    908   
2        29             192                6.620690                   1001   
3         3               5                1.666667                     85   
4         5              16                3.200000                    196   

   cluster             cluster_name recommended_zone  distance_from_packing  \
0        2     High-Volume Products           Zone A                     10   
1        0  Active/Regular Products           Zone B                     25   
2        0  Active/Regular Products           Zone B                     25   
3        1     Slow-Moving Products           Zone D                     60   
4        1     Slow-Moving Products           Zone D                     60   

   picking_priority  current_distance  distance_saved  o

In [88]:
print(product_features.index[:20])

RangeIndex(start=0, stop=20, step=1)


In [89]:
print(df[["StockCode", "InvoiceNo"]].head(20))

   StockCode InvoiceNo
0     85123A    536365
1      71053    536365
2     84406B    536365
3     84029G    536365
4     84029E    536365
5      22752    536365
6      21730    536365
7      22633    536366
8      22632    536366
9      84879    536367
10     22745    536367
11     22748    536367
12     22749    536367
13     22310    536367
14     84969    536367
15     22623    536367
16     22622    536367
17     21754    536367
18     21755    536367
19     21777    536367


In [90]:
print(product_features.shape)

(3922, 12)


In [91]:
print(product_features["StockCode"].head(20).tolist())

['71', '22', '29', '3', '5', '4', '91', '196', '175', '113', '12', '138', '518', '145', '102', '60', '88', '84', '321', '456']


In [92]:
product_base = df.groupby("StockCode").agg(
    total_quantity=("Quantity", "sum"),
    avg_quantity_per_order=("Quantity", "mean")
).reset_index()

In [93]:
product_base.head()

,StockCode,total_quantity,avg_quantity_per_order
0,10002,860,12.112676
1,10080,303,13.772727
2,10120,192,6.620690
3,10123C,5,1.666667
4,10124A,16,3.200000


In [94]:
order_frequency = (
    df.groupby("StockCode")["InvoiceNo"]
      .nunique()
      .reset_index(name="order_frequency")
)

In [95]:
product_base = product_base.merge(
    order_frequency,
    on="StockCode",
    how="left"
)

In [96]:
product_base.head()

,StockCode,total_quantity,avg_quantity_per_order,order_frequency
0,10002,860,12.112676,71
1,10080,303,13.772727,22
2,10120,192,6.620690,29
3,10123C,5,1.666667,3
4,10124A,16,3.200000,5


In [98]:
pairs_df = pd.read_csv(
    "../data/processed/product_pairs.csv"
)

In [99]:
print(pairs_df.head())
print(pairs_df.columns)

          Product_Pair  Order_Count   support
0  ('22386', '85099B')          825  0.041329
1   ('22697', '22699')          767  0.038423
2  ('21931', '85099B')          724  0.036269
3  ('22411', '85099B')          680  0.034065
4   ('20725', '22383')          655  0.032812
Index(['Product_Pair', 'Order_Count', 'support'], dtype='object')


In [100]:
import ast

related_products = {}

for pair in pairs_df["Product_Pair"]:
    try:
        product_a, product_b = ast.literal_eval(pair)

        product_a = str(product_a).strip()
        product_b = str(product_b).strip()

        related_products.setdefault(product_a, set()).add(product_b)
        related_products.setdefault(product_b, set()).add(product_a)

    except:
        pass

In [101]:
print(len(related_products))

3913


In [102]:
list(related_products.items())[:5]

[('22386',
  {'22156',
   '22352',
   '22581',
   '90039B',
   '23638',
   '85049F',
   '22431',
   '22090',
   '23004',
   '23335',
   '22561',
   '22959',
   '23009',
   '23281',
   '21792',
   '22631',
   '23156',
   '20622',
   '22766',
   '21061',
   '90040C',
   '22073',
   '21742',
   '21992',
   '72807c',
   '22149',
   '22271',
   '82613D',
   '22121',
   '22359',
   '22128',
   '22881',
   '21188',
   '85017A',
   '84944',
   '35818P',
   '23530',
   '84659A',
   '22311',
   '22857',
   '84251B',
   '21747',
   '21576',
   '23218',
   '22861',
   '21367',
   '90200C',
   '23598',
   '85126',
   '22815',
   '22846',
   '22110',
   '21332',
   '35650',
   '21181',
   '21390',
   '20782',
   '22935',
   '22873',
   '35967',
   '23178',
   '85205B',
   '22395',
   '21028',
   '22818',
   '22350',
   '22024',
   '22255',
   '90063B',
   '23043',
   '21243',
   '21678',
   '85132B',
   '84508A',
   '22883',
   '16048',
   '22523',
   '21866',
   '90055',
   '85132a',
   '22972',
  

In [103]:
product_base["StockCode"] = (
    product_base["StockCode"]
    .astype(str)
    .str.strip()
)

In [104]:
product_base["related_product_count"] = (
    product_base["StockCode"]
    .map(lambda x: len(related_products.get(x, set())))
)

In [105]:
product_base.head()

,StockCode,total_quantity,avg_quantity_per_order,order_frequency,related_product_count
0,10002,860,12.112676,71,2067
1,10080,303,13.772727,22,908
2,10120,192,6.620690,29,1001
3,10123C,5,1.666667,3,85
4,10124A,16,3.200000,5,196


In [106]:
clustering_features = product_base[
    [
        "StockCode",
        "order_frequency",
        "total_quantity",
        "avg_quantity_per_order",
        "related_product_count"
    ]
].copy()

In [107]:
clustering_features.head()

,StockCode,order_frequency,total_quantity,avg_quantity_per_order,related_product_count
0,10002,71,860,12.112676,2067
1,10080,22,303,13.772727,908
2,10120,29,192,6.620690,1001
3,10123C,3,5,1.666667,85
4,10124A,5,16,3.200000,196


In [108]:
from sklearn.preprocessing import StandardScaler

features_for_clustering = [
    "order_frequency",
    "total_quantity",
    "avg_quantity_per_order",
    "related_product_count"
]

X = product_base[features_for_clustering].copy()

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [109]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

product_base["cluster"] = kmeans.fit_predict(X_scaled)

In [110]:
print(product_base["cluster"].value_counts().sort_index())

cluster
0    2192
1    1551
2       1
3     178
Name: count, dtype: int64


In [111]:
cluster_summary = product_base.groupby("cluster")[
    features_for_clustering
].mean()

print(cluster_summary)

         order_frequency  total_quantity  avg_quantity_per_order  \
cluster                                                            
0             161.242245     1414.904653                8.246309   
1              17.390071      173.529336                8.247929   
2               1.000000    80995.000000            80995.000000   
3             781.949438    11914.662921               17.931036   

         related_product_count  
cluster                         
0                  2573.554288  
1                   827.516441  
2                     0.000000  
3                  3237.657303  


In [134]:
cluster_names = {
    0: "Active/Regular Products",
    1: "Slow-Moving Products",
    2: "Bulk/Exceptional Products",
    3: "High-Volume Products"
}

In [135]:
product_base["cluster_name"] = (
    product_base["cluster"].map(cluster_names)
)

In [136]:
print(
    product_base["cluster_name"].value_counts()
)

cluster_name
Active/Regular Products      2192
Slow-Moving Products         1551
High-Volume Products          178
Bulk/Exceptional Products       1
Name: count, dtype: int64


In [137]:
zone_mapping = {
    "High-Volume Products": "Zone A",
    "Active/Regular Products": "Zone B",
    "Low-Movement Products": "Zone C",
    "Slow-Moving Products": "Zone D"
}

product_base["recommended_zone"] = (
    product_base["cluster_name"].map(zone_mapping)
)

In [138]:
product_base[
    [
        "StockCode",
        "cluster_name",
        "recommended_zone"
    ]
].head(20)

,StockCode,cluster_name,recommended_zone
0,10002,Active/Regular Products,Zone B
1,10080,Slow-Moving Products,Zone D
2,10120,Slow-Moving Products,Zone D
3,10123C,Slow-Moving Products,Zone D
4,10124A,Slow-Moving Products,Zone D
5,10124G,Slow-Moving Products,Zone D
6,10125,Active/Regular Products,Zone B
7,10133,Active/Regular Products,Zone B
8,10135,Active/Regular Products,Zone B
9,11001,Active/Regular Products,Zone B


In [139]:
zone_distance = {
    "Zone A": 10,
    "Zone B": 25,
    "Zone C": 40,
    "Zone D": 60
}

product_base["distance_from_packing"] = (
    product_base["recommended_zone"].map(zone_distance)
)

In [140]:
product_base[
    [
        "StockCode",
        "recommended_zone",
        "distance_from_packing"
    ]
].head(20)

,StockCode,recommended_zone,distance_from_packing
0,10002,Zone B,25.0
1,10080,Zone D,60.0
2,10120,Zone D,60.0
3,10123C,Zone D,60.0
4,10124A,Zone D,60.0
5,10124G,Zone D,60.0
6,10125,Zone B,25.0
7,10133,Zone B,25.0
8,10135,Zone B,25.0
9,11001,Zone B,25.0


In [141]:
product_base["picking_priority"] = (
    product_base["order_frequency"]
    * product_base["total_quantity"]
)

In [142]:
top_products = product_base.sort_values(
    "picking_priority",
    ascending=False
)

top_products[
    [
        "StockCode",
        "order_frequency",
        "total_quantity",
        "cluster_name",
        "recommended_zone",
        "picking_priority"
    ]
].head(20)

,StockCode,order_frequency,total_quantity,cluster_name,recommended_zone,picking_priority
3387,85099B,2089,48371,High-Volume Products,Zone A,101047019
3407,85123A,2198,37641,High-Volume Products,Zone A,82734918
1109,22197,1392,56898,High-Volume Products,Zone A,79202016
3194,84879,1455,36362,High-Volume Products,Zone A,52906710
439,21212,1320,36396,High-Volume Products,Zone A,48042720
2670,47566,1685,18283,High-Volume Products,Zone A,30806855
1942,23084,994,30739,High-Volume Products,Zone A,30554566
175,20725,1565,19432,High-Volume Products,Zone A,30411080
2909,84077,535,54951,High-Volume Products,Zone A,29398785
1310,22423,1988,13851,High-Volume Products,Zone A,27535788


In [143]:
product_base["current_distance"] = 60

In [144]:
product_base["distance_saved"] = (
    product_base["current_distance"]
    - product_base["distance_from_packing"]
)

In [145]:
product_base[
    [
        "StockCode",
        "current_distance",
        "distance_from_packing",
        "distance_saved"
    ]
].head(20)

,StockCode,current_distance,distance_from_packing,distance_saved
0,10002,60,25.0,35.0
1,10080,60,60.0,0.0
2,10120,60,60.0,0.0
3,10123C,60,60.0,0.0
4,10124A,60,60.0,0.0
5,10124G,60,60.0,0.0
6,10125,60,25.0,35.0
7,10133,60,25.0,35.0
8,10135,60,25.0,35.0
9,11001,60,25.0,35.0


In [146]:
total_current_distance = (
    product_base["current_distance"]
    * product_base["order_frequency"]
).sum()

In [147]:
total_recommended_distance = (
    product_base["distance_from_packing"]
    * product_base["order_frequency"]
).sum()

In [148]:
print(
    "Current estimated travel:",
    total_current_distance
)

print(
    "Recommended estimated travel:",
    total_recommended_distance
)

Current estimated travel: 31176180
Recommended estimated travel: 11846265.0


In [149]:
if total_current_distance > 0:

    travel_reduction = (
        (total_current_distance - total_recommended_distance)
        / total_current_distance
    ) * 100

else:
    travel_reduction = 0

print(
    f"Estimated travel reduction: {travel_reduction:.2f}%"
)

Estimated travel reduction: 62.00%


In [150]:
final_recommendations = product_base[
    [
        "StockCode",
        "order_frequency",
        "total_quantity",
        "avg_quantity_per_order",
        "related_product_count",
        "cluster_name",
        "recommended_zone",
        "distance_from_packing",
        "picking_priority",
        "current_distance",
        "distance_saved"
    ]
].copy()

In [151]:
final_recommendations = final_recommendations.sort_values(
    "picking_priority",
    ascending=False
)

In [152]:
final_recommendations.head(20)

,StockCode,order_frequency,total_quantity,avg_quantity_per_order,related_product_count,cluster_name,recommended_zone,distance_from_packing,picking_priority,current_distance,distance_saved
3387,85099B,2089,48371,22.935514,3513,High-Volume Products,Zone A,10.0,101047019,60,50.0
3407,85123A,2198,37641,16.707057,3562,High-Volume Products,Zone A,10.0,82734918,60,50.0
1109,22197,1392,56898,40.125529,3545,High-Volume Products,Zone A,10.0,79202016,60,50.0
3194,84879,1455,36362,24.635501,3351,High-Volume Products,Zone A,10.0,52906710,60,50.0
439,21212,1320,36396,26.920118,3438,High-Volume Products,Zone A,10.0,48042720,60,50.0
2670,47566,1685,18283,10.761036,3480,High-Volume Products,Zone A,10.0,30806855,60,50.0
1942,23084,994,30739,30.225172,3122,High-Volume Products,Zone A,10.0,30554566,60,50.0
175,20725,1565,19432,12.283186,3469,High-Volume Products,Zone A,10.0,30411080,60,50.0
2909,84077,535,54951,102.520522,3125,High-Volume Products,Zone A,10.0,29398785,60,50.0
1310,22423,1988,13851,6.901345,3537,High-Volume Products,Zone A,10.0,27535788,60,50.0


In [153]:
final_recommendations.to_csv(
    "../data/processed/final_slot_recommendations.csv",
    index=False
)

In [154]:
final_recommendations[
    [
        "StockCode",
        "cluster_name",
        "recommended_zone",
        "picking_priority",
        "distance_saved"
    ]
].head(10)

,StockCode,cluster_name,recommended_zone,picking_priority,distance_saved
3387,85099B,High-Volume Products,Zone A,101047019,50.0
3407,85123A,High-Volume Products,Zone A,82734918,50.0
1109,22197,High-Volume Products,Zone A,79202016,50.0
3194,84879,High-Volume Products,Zone A,52906710,50.0
439,21212,High-Volume Products,Zone A,48042720,50.0
2670,47566,High-Volume Products,Zone A,30806855,50.0
1942,23084,High-Volume Products,Zone A,30554566,50.0
175,20725,High-Volume Products,Zone A,30411080,50.0
2909,84077,High-Volume Products,Zone A,29398785,50.0
1310,22423,High-Volume Products,Zone A,27535788,50.0


In [155]:
cluster_summary = product_base.groupby("cluster")[
    [
        "order_frequency",
        "total_quantity",
        "avg_quantity_per_order",
        "related_product_count"
    ]
].mean()

print(cluster_summary)

         order_frequency  total_quantity  avg_quantity_per_order  \
cluster                                                            
0             161.242245     1414.904653                8.246309   
1              17.390071      173.529336                8.247929   
2               1.000000    80995.000000            80995.000000   
3             781.949438    11914.662921               17.931036   

         related_product_count  
cluster                         
0                  2573.554288  
1                   827.516441  
2                     0.000000  
3                  3237.657303  
